# Produção de Petróleo, Líquidos de Gás Natural (LGN) e Gás Natural

**Autor:** Antonino Marques Jares

**Fonte:** Dados Abertos do GovBR

## Importações

In [83]:
import pandas as pd
from pathlib import Path

In [84]:
terminais = Path(rf"D:\ETL\nivel_01_bronze\capacidade_de_armazenagem_de_terminais\capacidade-armazenagem-terminais.csv")
df_terminais = pd.read_csv(terminais, sep=',', encoding='utf-8', low_memory=False)
df_terminais.head()

,Tipo,Municipio,UF,Operador,Numero_de_tanques,Capacidade_nominal_petroleo,Capacidade_nominal_derivados_biocombustiveis,Capacidade_nominal_GLP
0,FLUVIAL,VITORIA DO XINGU,PA,DORINALDO DA SILVA VITORIA DO XINGU,4.0,0.0,3323.0,0.0
1,FLUVIAL,SAO LUIS,MA,SANTOS BRASIL SÃO LUÍS 3,12.0,0.0,84567.0,0.0
2,FLUVIAL,SANTAREM,PA,ABI SANTARÉM,8.0,0.0,12806.0,0.0
3,FLUVIAL,PORTO VELHO,RO,ABI PORTO VELHO,12.0,0.0,16000.0,0.0
4,FLUVIAL,MANAUS,AM,NAVEMAZÔNIA MANAUS,0.0,0.0,0.0,0.0


# Renomeando Colunas

In [85]:

df_terminais = df_terminais.rename(columns={
    'Tipo': 'tipo',
    'Municipio': 'municipio',
    'UF': 'uf',
    'Operador': 'operador',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'Capacidade_nominal_petroleo': 'petroleo',
    'Capacidade_nominal_derivados_biocombustiveis': 'derivados_biocombustiveis',
    'Capacidade_nominal_GLP': 'glp'
})
df_terminais.head()

,tipo,municipio,uf,operador,Numero_de_tanques,petroleo,derivados_biocombustiveis,glp
0,FLUVIAL,VITORIA DO XINGU,PA,DORINALDO DA SILVA VITORIA DO XINGU,4.0,0.0,3323.0,0.0
1,FLUVIAL,SAO LUIS,MA,SANTOS BRASIL SÃO LUÍS 3,12.0,0.0,84567.0,0.0
2,FLUVIAL,SANTAREM,PA,ABI SANTARÉM,8.0,0.0,12806.0,0.0
3,FLUVIAL,PORTO VELHO,RO,ABI PORTO VELHO,12.0,0.0,16000.0,0.0
4,FLUVIAL,MANAUS,AM,NAVEMAZÔNIA MANAUS,0.0,0.0,0.0,0.0


## Retirando Colunas

In [86]:
df_terminais.drop(['Numero_de_tanques'], axis=1, inplace=True)
df_terminais.head()

,tipo,municipio,uf,operador,petroleo,derivados_biocombustiveis,glp
0,FLUVIAL,VITORIA DO XINGU,PA,DORINALDO DA SILVA VITORIA DO XINGU,0.0,3323.0,0.0
1,FLUVIAL,SAO LUIS,MA,SANTOS BRASIL SÃO LUÍS 3,0.0,84567.0,0.0
2,FLUVIAL,SANTAREM,PA,ABI SANTARÉM,0.0,12806.0,0.0
3,FLUVIAL,PORTO VELHO,RO,ABI PORTO VELHO,0.0,16000.0,0.0
4,FLUVIAL,MANAUS,AM,NAVEMAZÔNIA MANAUS,0.0,0.0,0.0


## Reordenando

In [87]:
df_terminais = df_terminais[['uf', 'municipio', 'tipo', 'operador','petroleo', 'derivados_biocombustiveis', 'glp']]
df_terminais.head()

,uf,municipio,tipo,operador,petroleo,derivados_biocombustiveis,glp
0,PA,VITORIA DO XINGU,FLUVIAL,DORINALDO DA SILVA VITORIA DO XINGU,0.0,3323.0,0.0
1,MA,SAO LUIS,FLUVIAL,SANTOS BRASIL SÃO LUÍS 3,0.0,84567.0,0.0
2,PA,SANTAREM,FLUVIAL,ABI SANTARÉM,0.0,12806.0,0.0
3,RO,PORTO VELHO,FLUVIAL,ABI PORTO VELHO,0.0,16000.0,0.0
4,AM,MANAUS,FLUVIAL,NAVEMAZÔNIA MANAUS,0.0,0.0,0.0


## Lendo json (baixar o minicipios_brasil_coordenadas.json)

In [88]:
import pandas as pd
import json
import unicodedata


caminho = "municipios_brasil_coordenadas.json"

# Lê o arquivo local
with open(caminho, "r", encoding="utf-8") as f:
    dados = json.load(f)

# Achata: cada município vira uma linha
df_municipios_geolocalizacao = pd.json_normalize(
    dados["unidades_federativas"],
    record_path="municipios",
    meta=["uf", "unidade_federativa", "municipio_capital"],
)

def normaliza_municipio(texto):
    if pd.isna(texto):
        return texto
    # Decompõe caracteres acentuados: 'á' -> 'a' + acento
    texto = unicodedata.normalize("NFKD", str(texto))
    # Remove os acentos (caracteres combinantes)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    # Troca 'ç' por 'c' (o NFKD não decompõe o ç, então tratamos à parte)
    texto = texto.replace("ç", "c").replace("Ç", "C")
    # Maiúsculas e remove espaços extras
    return texto.upper().strip()

df_municipios_geolocalizacao["municipio"] = df_municipios_geolocalizacao["municipio"].apply(normaliza_municipio)
df_municipios_geolocalizacao = df_municipios_geolocalizacao[['uf', 'municipio', 'latitude', 'longitude']]
print(df_municipios_geolocalizacao.shape)      # esperado: (5571, 7)
print(df_municipios_geolocalizacao.head())

(5571, 4)
   uf     municipio  latitude  longitude
0  AC    ACRELANDIA  -9.82581   -66.8972
1  AC  ASSIS BRASIL -10.92980   -69.5738
2  AC     BRASILEIA -10.99500   -68.7497
3  AC        BUJARI  -9.81528   -67.9550
4  AC      CAPIXABA -10.56600   -67.6860


# GEOLOCALIZAÇÂO

In [89]:
import re
import pandas as pd

# ============================================================
# 1) Mapa: apenas vogais acentuadas + ç
# ============================================================
MAPA = {
    "á": "a", "à": "a", "â": "a", "ã": "a", "ä": "a",
    "Á": "A", "À": "A", "Â": "A", "Ã": "A", "Ä": "A",
    "é": "e", "è": "e", "ê": "e", "ë": "e",
    "É": "E", "È": "E", "Ê": "E", "Ë": "E",
    "í": "i", "ì": "i", "î": "i", "ï": "i",
    "Í": "I", "Ì": "I", "Î": "I", "Ï": "I",
    "ó": "o", "ò": "o", "ô": "o", "õ": "o", "ö": "o",
    "Ó": "O", "Ò": "O", "Ô": "O", "Õ": "O", "Ö": "O",
    "ú": "u", "ù": "u", "û": "u", "ü": "u",
    "Ú": "U", "Ù": "U", "Û": "U", "Ü": "U",
    "ç": "c", "Ç": "C",
}

_PATTERN = re.compile("|".join(map(re.escape, MAPA.keys())))


def normaliza_municipio(texto):
    """Remove acentos de vogais, troca ç->c, colapsa espaços e deixa em maiúsculas."""
    if pd.isna(texto):
        return texto
    texto = _PATTERN.sub(lambda m: MAPA[m.group(0)], str(texto))
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto.upper()


# ============================================================
# 2) Aplica a normalização no df_terminais
# ============================================================
df_terminais = df_terminais.copy()
df_terminais["municipio_norm"] = df_terminais["municipio"].apply(normaliza_municipio)
df_terminais["uf_norm"] = df_terminais["uf"].astype(str).str.upper().str.strip()

# ============================================================
# 3) Aplica a normalização no df_municipios_geolocalizacao
# ============================================================
df_municipios_geolocalizacao = df_municipios_geolocalizacao.copy()
df_municipios_geolocalizacao["municipio_norm"] = (
    df_municipios_geolocalizacao["municipio"].apply(normaliza_municipio)
)
df_municipios_geolocalizacao["uf_norm"] = (
    df_municipios_geolocalizacao["uf"].astype(str).str.upper().str.strip()
)

# ============================================================
# 4) Merge por UF + município normalizado
# ============================================================
df_terminais = df_terminais.merge(
    df_municipios_geolocalizacao[
        ["uf_norm", "municipio_norm", "latitude", "longitude"]
    ],
    on=["uf_norm", "municipio_norm"],
    how="left",
)

# ============================================================
# 5) Remove colunas auxiliares
# ============================================================
df_terminais = df_terminais.drop(
    columns=["uf_norm", "municipio_norm"], errors="ignore"
)

# ============================================================
# 6) Diagnóstico
# ============================================================
sem_coord = df_terminais["latitude"].isna().sum()
print(f"Terminais sem coordenada: {sem_coord} de {len(df_terminais)}")

if sem_coord:
    print("\nMunicípios não encontrados:")
    print(
        df_terminais[df_terminais["latitude"].isna()][["uf", "municipio"]]
        .drop_duplicates()
        .sort_values(["uf", "municipio"])
        .to_string(index=False)
    )

print(df_terminais.head())

Terminais sem coordenada: 0 de 141
   uf         municipio     tipo                             operador  \
0  PA  VITORIA DO XINGU  FLUVIAL  DORINALDO DA SILVA VITORIA DO XINGU   
1  MA          SAO LUIS  FLUVIAL             SANTOS BRASIL SÃO LUÍS 3   
2  PA          SANTAREM  FLUVIAL                         ABI SANTARÉM   
3  RO       PORTO VELHO  FLUVIAL                      ABI PORTO VELHO   
4  AM            MANAUS  FLUVIAL                   NAVEMAZÔNIA MANAUS   

   petroleo  derivados_biocombustiveis  glp  latitude  longitude  
0       0.0                     3323.0  0.0  -2.87922   -52.0088  
1       0.0                    84567.0  0.0  -2.53874   -44.2825  
2       0.0                    12806.0  0.0  -2.43849   -54.6996  
3       0.0                    16000.0  0.0  -8.76077   -63.8999  
4       0.0                        0.0  0.0  -3.11866   -60.0212  


In [92]:
import json
from datetime import datetime
import pandas as pd

# ------------------------------------------------------------
# 1) Prepara os dados (DataFrame -> lista de registros)
# ------------------------------------------------------------
registros = df_terminais.to_dict(orient="records")

# Converte tipos numpy para tipos nativos do Python
def _nativo(v):
    if pd.isna(v):
        return None
    if hasattr(v, "item"):          # numpy scalar (int64, float64, etc.)
        return v.item()
    return v

registros = [
    {k: _nativo(v) for k, v in reg.items()}
    for reg in registros
]

# ------------------------------------------------------------
# 2) Monta a estrutura final
# ------------------------------------------------------------
saida = {
    "metadados": {
        "data_ultima_atualizacao": datetime.now().strftime("%d-%m-%Y"),
        "responsavel": "Antonino Marques Jares",
        "fonte": "https://dados.gov.br/dados/conjuntos-dados/instalacoes-portuarias",
        "orgao": "CGU",
    },
    "dados": registros,
}

# ------------------------------------------------------------
# 3) Grava o arquivo
# ------------------------------------------------------------
with open("armazens_combustiveis.json", "w", encoding="utf-8") as f:
    json.dump(saida, f, ensure_ascii=False, indent=2)

print(f"Arquivo gerado com {len(registros)} registros.")
print("Data de atualização:", saida["metadados"]["data_ultima_atualizacao"])

Arquivo gerado com 141 registros.
Data de atualização: 31-10-2026
